# ML Development Notebook for DysCalc

## Configs

### Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from scipy import stats
from dataclasses import dataclass
from typing import Literal, Optional, Any, Dict
from sklearn.model_selection import train_test_split

### ML Model Configs

## Functions and Classes

### Utility Functions

In [2]:
def stratified_sampler(dataframe: pd.DataFrame, label_col: str, split: list[float], random_state: int = 42) -> Dict[str, pd.DataFrame]:
    assert len(split) == 3, "split must be [train, test, val]"

    df = dataframe.copy()

    class_one_df = df[df[label_col] == 1]
    class_zero_df = df[df[label_col] == 0]

    class_one_train = class_one_df.sample(frac=split[0], random_state=random_state)
    class_zero_train = class_zero_df.sample(frac=split[0], random_state=random_state)
    train_df = pd.concat([class_one_train, class_zero_train]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    class_one_rem = class_one_df.drop(class_one_train.index)
    class_zero_rem = class_zero_df.drop(class_zero_train.index)

    test_frac = split[1] / (split[1] + split[2])
    class_one_test = class_one_rem.sample(frac=test_frac, random_state=random_state)
    class_zero_test = class_zero_rem.sample(frac=test_frac, random_state=random_state)
    test_df = pd.concat([class_one_test, class_zero_test]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    class_one_val = class_one_rem.drop(class_one_test.index)
    class_zero_val = class_zero_rem.drop(class_zero_test.index)
    val_df = pd.concat([class_one_val, class_zero_val]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    return {
        'train': train_df,
        'test': test_df,
        'val': val_df
    }

def split_xy(df: pd.DataFrame, label_col:str):
    X = df.drop(columns=[label_col])
    y = df[label_col]
    return X, y

### Node Class
Description for each Node in the tree

In [3]:
from Dataclasses import Node

### C4.5 Decision Tree Class Implementation

In [4]:
from C45DecisionTree import C45DecisionTree

## Training

### Data Loading & Splitting
Load the complete dataset vector and split it into Train (70%), Validation (15%), and Test (15%).

In [5]:
print("Loading GAN-Balanced 70/15/15 TSTR Datasets...")

train_df = pd.read_csv('dataset/FUNADB_balanced_TRAIN.csv')
val_df = pd.read_csv('dataset/FUNADB_real_VAL.csv')
test_df = pd.read_csv('dataset/FUNADB_real_TEST.csv')

print(f"Train shape (Balanced 70%): {train_df.shape}")
print(f"Validation shape (Purely Real 15%): {val_df.shape}")
print(f"Test shape (Purely Real 15%): {test_df.shape}")

X_train, y_train = split_xy(train_df, 'Label')
X_val, y_val = split_xy(val_df, 'Label')
X_test, y_test = split_xy(test_df, 'Label')

Loading GAN-Balanced 70/15/15 TSTR Datasets...
Train shape (Balanced 70%): (308, 13)
Validation shape (Purely Real 15%): (54, 13)
Test shape (Purely Real 15%): (54, 13)


### Cross-Validation & Hyperparameter Tuning
Perform Grid Search with 5-fold Stratified Cross-Validation on the training set to optimize `conf_fact`, `min_samples_leaf`, and `max_depth`.

In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score, roc_auc_score
import itertools

def evaluate_model(y_true, y_pred, y_prob=None):
    metrics = {
        'Recall': recall_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred),
        'Accuracy': accuracy_score(y_true, y_pred)
    }
    if y_prob is not None:
        metrics['AUC'] = roc_auc_score(y_true, y_prob)
    return metrics

# Expanded parameter grid for better coverage
param_grid = {
    'conf_fact': [0.05, 0.10, 0.15, 0.25, 0.35, 0.45],  # Added lower values for less aggressive pruning
    'min_samples_leaf': [1, 2, 3, 5, 8],  # Added smaller values for more flexibility
    'max_depth': [None, 7, 10, 12, 15]  # Added None (unlimited) and higher depths
}

# Feature domain mapping for diagnostic outputs
domain_mapping = {
    'NC': 'Number Processing',
    'DM': 'Number Processing',
    'NS': 'Arithmetic Fluency',
    'ADD': 'Arithmetic Fluency',
    'SUB': 'Arithmetic Fluency',
    'CA': 'Arithmetic Fluency',
    'NP': 'Derived Cognitive',
    'SN': 'Derived Cognitive',
    'AF': 'Derived Cognitive',
    'BC': 'Derived Cognitive',
    'AS': 'Derived Cognitive',
    'PF': 'Derived Cognitive'
}

keys, values = zip(*param_grid.items())
hyperparams_combos = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Total hyperparameter combinations: {len(hyperparams_combos)}")

# Track all results for analysis
all_results = []
best_f1 = -1
best_params = None
best_cv_metrics = None

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Starting Grid Search CV...")
for idx, params in enumerate(hyperparams_combos, 1):
    cv_f1 = []
    cv_recall = []
    cv_precision = []
    cv_accuracy = []
    
    for train_index, val_index in skf.split(X_train, y_train):
        # Manual K-Fold splits (remember indices in current fold)
        X_trn_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_trn_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
        
        tree = C45DecisionTree(**params, feature_domain_mapping=domain_mapping)
        tree.fit(X_trn_fold, y_trn_fold)
        
        # FIX: Extract probabilities and apply 0.35 threshold during training evaluation
        SEARCH_THRESHOLD = 0.375
        diagnostics = tree.predict_with_diagnostics(X_val_fold)
        probs = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence for d in diagnostics]
        preds = [1 if p >= SEARCH_THRESHOLD else 0 for p in probs]
        
        cv_f1.append(f1_score(y_val_fold, preds))
        cv_recall.append(recall_score(y_val_fold, preds))
        cv_precision.append(precision_score(y_val_fold, preds, zero_division=0))
        cv_accuracy.append(accuracy_score(y_val_fold, preds))
        
    mean_f1 = np.mean(cv_f1)
    mean_recall = np.mean(cv_recall)
    std_f1 = np.std(cv_f1)
    std_recall = np.std(cv_recall)
    
    # Store all results for analysis
    all_results.append({
        'params': params,
        'mean_recall': mean_recall,
        'mean_f1': mean_f1,
        'mean_precision': np.mean(cv_precision),
        'mean_accuracy': np.mean(cv_accuracy)
    })
    
    # Progress reporting every 50 combinations
    if idx % 50 == 0:
        print(f"Evaluated {idx}/{len(hyperparams_combos)} combinations...")
    
    # Optimizing for F1-score with minimum Recall >= 0.85
    if mean_recall >= 0.85 and mean_f1 > best_f1:
        best_f1 = mean_f1
        best_params = params
        best_cv_metrics = {
            'mean_f1': mean_f1,
            'std_f1': std_f1,
            'mean_recall': mean_recall,
            'std_recall': std_recall,
            'mean_precision': np.mean(cv_precision),
            'std_precision': np.std(cv_precision),
            'mean_accuracy': np.mean(cv_accuracy),
            'std_accuracy': np.std(cv_accuracy)
        }

# Analyze results
valid_results = [r for r in all_results if r['mean_recall'] >= 0.85]

if valid_results:
    all_results_sorted = sorted(valid_results, key=lambda x: x['mean_f1'], reverse=True)
else:
    all_results_sorted = sorted(all_results, key=lambda x: x['mean_recall'], reverse=True) 

max_recall_achieved = all_results_sorted[0]['mean_recall']

print(f"\n=== Grid Search Summary ===")
print(f"Maximum recall achieved: {max_recall_achieved:.4f}")
print(f"\nTop 5 configurations by recall:")
for i, result in enumerate(all_results_sorted[:5], 1):
    print(f"{i}. Recall={result['mean_recall']:.4f}, F1={result['mean_f1']:.4f}, Params={result['params']}")
        
if best_params is None:
    print("\nWarning: No hyperparameter combination achieved recall >= 0.85.")
    
    # If close to threshold, use best recall config
    if max_recall_achieved >= 0.80:
        print(f"Using configuration with highest recall ({max_recall_achieved:.4f})...")
        best_result = all_results_sorted[0]
        best_params = best_result['params']
        
        # Recalculate full CV metrics for best params
        cv_f1, cv_recall, cv_precision, cv_accuracy = [], [], [], []
        for train_index, val_index in skf.split(X_train, y_train):
            X_trn_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_trn_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
            
            tree = C45DecisionTree(**best_params, feature_domain_mapping=domain_mapping)
            tree.fit(X_trn_fold, y_trn_fold)

            diagnostics = tree.predict_with_diagnostics(X_val_fold)
            probs = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence for d in diagnostics]
            preds = [1 if p >= 0.375 else 0 for p in probs]
            
            cv_f1.append(f1_score(y_val_fold, preds))
            cv_recall.append(recall_score(y_val_fold, preds))
            cv_precision.append(precision_score(y_val_fold, preds, zero_division=0))
            cv_accuracy.append(accuracy_score(y_val_fold, preds))
        
        best_cv_metrics = {
            'mean_f1': np.mean(cv_f1),
            'std_f1': np.std(cv_f1),
            'mean_recall': np.mean(cv_recall),
            'std_recall': np.std(cv_recall),
            'mean_precision': np.mean(cv_precision),
            'std_precision': np.std(cv_precision),
            'mean_accuracy': np.mean(cv_accuracy),
            'std_accuracy': np.std(cv_accuracy)
        }
    else:
        print(f"Maximum recall ({max_recall_achieved:.4f}) is below 0.80.")
        print("Consider: 1) Feature engineering, 2) Class balancing, 3) Different algorithm")
        print("Proceeding with highest recall configuration for now...")
        best_params = all_results_sorted[0]['params']
    
print(f"\n=== Best Hyperparameters ===")
print(f"Confidence Factor: {best_params['conf_fact']}")
print(f"Min Samples Leaf: {best_params['min_samples_leaf']}")
print(f"Max Depth: {best_params['max_depth']}")

if best_cv_metrics is not None:
    print(f"\n=== Cross-Validation Performance (5-Fold) ===")
    print(f"Recall:    {best_cv_metrics['mean_recall']:.4f} ± {best_cv_metrics['std_recall']:.4f}")
    print(f"Precision: {best_cv_metrics['mean_precision']:.4f} ± {best_cv_metrics['std_precision']:.4f}")
    print(f"F1-Score:  {best_cv_metrics['mean_f1']:.4f} ± {best_cv_metrics['std_f1']:.4f}")
    print(f"Accuracy:  {best_cv_metrics['mean_accuracy']:.4f} ± {best_cv_metrics['std_accuracy']:.4f}")

Total hyperparameter combinations: 150
Starting Grid Search CV...
Evaluated 50/150 combinations...
Evaluated 100/150 combinations...
Evaluated 150/150 combinations...

=== Grid Search Summary ===
Maximum recall achieved: 0.8581

Top 5 configurations by recall:
1. Recall=0.8581, F1=0.6360, Params={'conf_fact': 0.45, 'min_samples_leaf': 2, 'max_depth': 7}

=== Best Hyperparameters ===
Confidence Factor: 0.45
Min Samples Leaf: 2
Max Depth: 7

=== Cross-Validation Performance (5-Fold) ===
Recall:    0.8581 ± 0.2680
Precision: 0.5920 ± 0.1589
F1-Score:  0.6360 ± 0.0805
Accuracy:  0.5484 ± 0.0518


### Final Model Training & Evaluation
Train the optimal model on the entire training set and evaluate on Validation and Test sets.

In [26]:
final_tree = C45DecisionTree(**best_params, feature_domain_mapping=domain_mapping)
final_tree.fit(X_train, y_train)
'''
print("--- Scanning for Optimal Threshold (>85% Recall) ---")
val_diagnostics_scan = final_tree.predict_with_diagnostics(X_val)
val_probs_scan = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence for d in val_diagnostics_scan]

best_thresh = 0.15 # Fallback

for thresh in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15]:
    val_preds_scan = [1 if prob >= thresh else 0 for prob in val_probs_scan]
    if recall_score(y_val, val_preds_scan) >= 0.85:
        best_thresh = thresh
        print(f"Target Unlocked! Setting operational threshold to: {best_thresh}")
        break

THRESHOLD = best_thresh
'''
THRESHOLD = 0.375

print(f"\n--- Validation Set Performance (Threshold: {THRESHOLD}) ---")
val_diagnostics = final_tree.predict_with_diagnostics(X_val)
val_probs = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence for d in val_diagnostics]
val_preds_adjusted = [1 if prob >= THRESHOLD else 0 for prob in val_probs]
val_metrics = evaluate_model(y_val, val_preds_adjusted, val_probs)
for k, v in val_metrics.items():
    print(f"{k}: {v:.4f}")

print(f"\n--- Test Set Performance (Threshold: {THRESHOLD}) ---")
test_diagnostics = final_tree.predict_with_diagnostics(X_test)
test_probs = [d.confidence if int(d.predicted_class) == 1 else 1 - d.confidence for d in test_diagnostics]
test_preds_adjusted = [1 if prob >= THRESHOLD else 0 for prob in test_probs]
test_metrics = evaluate_model(y_test, test_preds_adjusted, test_probs)
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")


--- Validation Set Performance (Threshold: 0.375) ---
Recall: 1.0000
Precision: 0.4038
F1-Score: 0.5753
Accuracy: 0.4259
AUC: 0.7763

--- Test Set Performance (Threshold: 0.375) ---
Recall: 0.9524
Precision: 0.3774
F1-Score: 0.5405
Accuracy: 0.3704
AUC: 0.6169


### Interpretability & Diagnostic Outputs
Extract Global Feature Importance and generate a sample explanation for an 'At-Risk' prediction.

In [8]:
print("\n--- Global Feature Importance ---")
importance = final_tree.get_feature_importance()
sorted_importance = sorted(importance.items(), key=lambda item: item[1], reverse=True)
for feature, val in sorted_importance:
    print(f"{feature}: {val:.4f}")

print("\n--- Sample Diagnostic Output ('At-Risk' Case) ---")
for idx, diag in enumerate(test_diagnostics):
    if diag.predicted_class == 1:  # Represents 'At-Risk'
        print(f"Test Case #{idx}:")
        print("  Predicted Class: At-Risk (1)")
        print(f"  Confidence: {diag.confidence:.4f}")
        print(f"  Decision Path: {diag.decision_path_readable}")
        print(f"  Domain Deficit Severity: {diag.domain_severity_scores}")


--- Global Feature Importance ---
PF: 0.3666
BC: 0.2946
AF: 0.1376
SN: 0.1288
NS: 0.1140

--- Sample Diagnostic Output ('At-Risk' Case) ---
Test Case #12:
  Predicted Class: At-Risk (1)
  Confidence: 1.0000
  Decision Path: PF <= 0.0058
  Domain Deficit Severity: {'Arithmetic Fluency': 0.0, 'Number Processing': 0.0, 'Derived Cognitive': np.float64(0.2370335142977757)}
Test Case #16:
  Predicted Class: At-Risk (1)
  Confidence: 1.0000
  Decision Path: PF <= 0.0058
  Domain Deficit Severity: {'Arithmetic Fluency': 0.0, 'Number Processing': 0.0, 'Derived Cognitive': np.float64(0.2370335142977757)}
Test Case #20:
  Predicted Class: At-Risk (1)
  Confidence: 1.0000
  Decision Path: PF <= 0.0058
  Domain Deficit Severity: {'Arithmetic Fluency': 0.0, 'Number Processing': 0.0, 'Derived Cognitive': np.float64(0.2370335142977757)}
Test Case #24:
  Predicted Class: At-Risk (1)
  Confidence: 1.0000
  Decision Path: PF > 0.0058 AND BC > -0.0004 AND AF > 11.9637 AND SN > -2779.9789 AND PF <= 0.0532

In [9]:
print("\n--- Serializing Final Model ---")

final_tree.save_model(
    filepath='dyscalc_final_model.pkl', 
    optimal_threshold=THRESHOLD  # Uses the discovered threshold
)


--- Serializing Final Model ---
Model successfully saved to dyscalc_final_model.pkl
Locked pedagogical threshold: 0.35
